In [1]:
import numpy as np

In [2]:
def get_disp(coords, latt):
   """
   Calculate displacements for NVT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array of with shape [site, time step, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   d_coords = coords[:, 1:] - coords[:, :-1]
   d_coords = d_coords - np.round(d_coords)
   f_disp = np.cumsum(d_coords, axis=1)
   latt = np.array(latt)
   disp = np.einsum('ijk,jkl->ijk', f_disp, latt[1:])
   return disp


def get_disp_npt(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt)
   wrapped_diff = np.diff(wrapped, axis=1)
   latt_para = np.einsum('ijj->ij', latt)

   unwrapped_disp = wrapped_diff - (np.floor(wrapped_diff / latt_para[1:] + 1 / 2) * latt_para[1:])

   return np.cumsum(unwrapped_disp, axis=1)


def get_disp_npt_matrix(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   latt_inv = np.linalg.inv(latt)
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt)
   wrapped_diff = np.diff(wrapped, axis=1)

   unwrapped_disp = wrapped_diff - (np.einsum('ijk,jkl->ijk', np.floor(np.einsum('ijk,jkl->ijk', wrapped_diff, latt_inv[1:]) + (1 / 2)), latt[1:]))

   return np.cumsum(unwrapped_disp, axis=1)

In [3]:
coords = np.array([[[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.2,0.2,0.2],[0.2,0.2,0.2],[0.2,0.2,0.2],[0.2,0.2,0.2]],
                   [[0.3,0.2,0.2],[0.3,0.2,0.2],[0.3,0.2,0.2],[0.3,0.2,0.2]],
                   [[0.4,0.2,0.2],[0.4,0.2,0.2],[0.4,0.2,0.2],[0.4,0.2,0.2]]
                   ])
coords = np.expand_dims(coords, 2)

latt = np.array([[[10,2,2],[2,10,2],[2,2,10]],
                 [[10,2,2],[2,10,2],[2,2,10]],
                 [[10,2,2],[2,10,2],[2,2,10]],
                 [[10,2,2],[2,10,2],[2,2,10]],
                 [[10,2,2],[2,10,2],[2,2,10]],
                ])

print(coords.shape, latt.shape)

(5, 4, 1, 3) (5, 3, 3)


In [4]:
nvt_disp = get_disp(coords, latt)
npt_disp = get_disp_npt(coords, latt)
npt_matrix_disp = get_disp_npt_matrix(coords, latt)

np.testing.assert_array_almost_equal(npt_matrix_disp, npt_disp)
np.testing.assert_array_almost_equal(npt_disp, nvt_disp)

In [5]:
for i in range(100):
   rand_coords = np.random.rand(5,4,1,3)
   rand_latt = np.random.rand(5,3,3)

   nvt_disp = get_disp(coords, latt)
   npt_disp = get_disp_npt(coords, latt)
   npt_matrix_disp = get_disp_npt_matrix(coords, latt)

   np.testing.assert_array_almost_equal(npt_matrix_disp, npt_disp)
   np.testing.assert_array_almost_equal(npt_disp, nvt_disp)

In [6]:
print(get_disp(coords, latt))

[[[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]]


In [7]:
print(get_disp_npt(coords, latt))

[[[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]]


In [8]:
print(get_disp_npt_matrix(coords, latt))

[[[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]

 [[0.  0.  0. ]
  [1.4 1.4 1.4]
  [2.8 1.4 1.4]
  [4.2 1.4 1.4]]]
